In [ ]:
import os
import re

BASE_PATH = "C:/Users/panaa/Desktop/dipseer"

TRAIN_DIR = os.path.join(BASE_PATH, "train")
VAL_DIR = os.path.join(BASE_PATH, "val")
TEST_DIR = os.path.join(BASE_PATH, "test")

DATASET_SPLITS = [TRAIN_DIR, VAL_DIR, TEST_DIR]

# Filename pattern for metadata JSON files (HH_MM_SS_microseconds.json)
METADATA_REGEX = re.compile(r"^(\d{2})_(\d{2})_(\d{2})_(\d{6})\.json$", re.IGNORECASE)

In [ ]:
from datetime import datetime


def parse_metadata_timestamp(filename):
    """
    Parses a metadata filename into a datetime object.
    """

    match = METADATA_REGEX.match(filename)
    if not match:
        return None

    hh, mm, ss, micros = match.groups()

    try:
        return datetime(2000, 1, 1, int(hh), int(mm), int(ss), int(micros))
    except ValueError:
        return None


def parse_label_timestamp(timestamp_str):
    """
    Parses a label timestamp into a datetime object.
    """

    try:
        hh, mm, ss, micros = timestamp_str.strip().split(":")
        return datetime(2000, 1, 1, int(hh), int(mm), int(ss), int(micros))
    except (ValueError, AttributeError):
        return None

In [ ]:
import json

def load_json(path):
    """
    Loads a JSON file and returns its content.
    """

    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except (OSError, json.JSONDecodeError):
        return None


def save_json(path, data):
    """
    Writes data to a JSON file.
    """

    try:
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=4, ensure_ascii=False)
    except OSError:
        pass


In [ ]:
def find_subject_directories(root_dir):
    """
    Finds directories that contain both 'metadata' and 'labels' subdirectories.
    """

    subject_dirs = []

    for current_root, dirs, _ in os.walk(root_dir):
        dirs_lower = {d.lower() for d in dirs}
        if "metadata" in dirs_lower and "labels" in dirs_lower:
            subject_dirs.append(current_root)

    return subject_dirs

In [ ]:
def load_metadata_entries(metadata_dir):
    """
    Loads valid metadata JSON files from a directory and returns them sorted by timestamp.
    """

    entries = []

    try:
        filenames = os.listdir(metadata_dir)
    except OSError:
        return []

    for filename in filenames:
        path = os.path.join(metadata_dir, filename)

        if not os.path.isfile(path) or not filename.endswith(".json"):
            continue

        timestamp = parse_metadata_timestamp(filename)
        if timestamp is None:
            continue

        entries.append((filename, path, timestamp))

    entries.sort(key=lambda x: x[2])
    return entries

In [ ]:
def load_label_files(labels_dir):
    """
    Loads all JSON label files from a directory and returns mapping: {labeler_name: file_path}.
    """

    label_files = {}

    try:
        filenames = os.listdir(labels_dir)
    except OSError:
        return label_files

    for filename in filenames:
        path = os.path.join(labels_dir, filename)

        if os.path.isfile(path) and filename.endswith(".json"):
            labeler_name = os.path.splitext(filename)[0]
            label_files[labeler_name] = path

    return label_files

In [ ]:
def build_timeline(label_path):
    """
    Builds temporal timelines for attention and emotion from a label JSON file.
    """

    data = load_json(label_path)
    if not isinstance(data, list):
        return None

    attention, emotion = [], []

    for item in data:
        if not isinstance(item, dict):
            continue

        timestamp = parse_label_timestamp(item.get("datetime", ""))
        if timestamp is None:
            continue

        if "attention" in item:
            try:
                attention.append((timestamp, int(item["attention"])))
            except(ValueError, TypeError):
                pass

        if "emotion" in item:
            try:
                emotion.append((timestamp, int(item["emotion"])))
            except(ValueError, TypeError):
                pass

    attention.sort()
    emotion.sort()

    return {
        "attention_times": [t for t, _ in attention],
        "attention_values": [v for _, v in attention],
        "emotion_times": [t for t, _ in emotion],
        "emotion_values": [v for _, v in emotion],
    }

In [ ]:
from bisect import bisect_right

def get_latest_value(target, times, values):
    """
    Returns the latest value whose timestamp is less or equal to target.
    """

    if not times:
        return None

    idx = bisect_right(times, target) - 1
    return values[idx] if idx >= 0 else None


def build_labels(metadata_time, timelines):
    """
    Builds label values for a given metadata timestamp using fill-forward.
    """

    result = {}

    for name, timeline in timelines.items():
        result[name] = {
            "attention": get_latest_value(metadata_time, timeline["attention_times"], timeline["attention_values"]),
            "emotion": get_latest_value(metadata_time, timeline["emotion_times"], timeline["emotion_values"]),
        }

    return result

In [ ]:
def process_subject(subject_dir):
    """
    Processes a subject directory by attaching labels to each metadata file.
    """

    metadata_dir = os.path.join(subject_dir, "metadata")
    labels_dir = os.path.join(subject_dir, "labels")

    metadata_entries = load_metadata_entries(metadata_dir)
    label_files = load_label_files(labels_dir)

    if not metadata_entries or not label_files:
        return 0, 0

    timelines = {}
    for labeler_name, path in label_files.items():
        timeline = build_timeline(path)
        if timeline:
            timelines[labeler_name] = timeline

    if not timelines:
        return 0, 0

    updated_count = 0

    for _, path, timestamp in metadata_entries:
        data = load_json(path)
        if not isinstance(data, dict):
            continue

        data["labels"] = build_labels(timestamp, timelines)

        save_json(path, data)
        updated_count += 1

    return len(metadata_entries), updated_count

In [ ]:
def process_dataset(root_dir):
    total_files, updated_files = 0, 0

    for subject_dir in find_subject_directories(root_dir):
        files_count, updated_count = process_subject(subject_dir)
        total_files += files_count
        updated_files += updated_count

    return total_files, updated_files

In [ ]:
def run():
    total_files, updated_files = 0, 0

    for dataset in DATASET_SPLITS:
        files_count, updated_count = process_dataset(dataset)
        total_files += files_count
        updated_files += updated_count

    print(f"Finished: {updated_files}/{total_files} metadata files updated.")

In [ ]:
if __name__ == "__main__":
    run()